In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

In [ ]:
df = pd.read_csv("/kaggle/input/datasets/rohan0301/unsupervised-learning-on-country-data/Country-data.csv")
print(df.shape)
df.head()

Data Cleaning 

In [ ]:
df.columns = df.columns.str.strip()
before = df.shape[0]
df = df.drop_duplicates()
print(f"Dropped {before - df.shape[0]} duplicate rows")

numeric_cols = df.columns.drop("country")
df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric, errors="coerce")

for col in numeric_cols:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].median())

print(df.isnull().sum())
df.describe()

Feature Seperation and Scaling.

In [ ]:
country_names = df["country"]
features = df.drop(columns=["country"])

scaler = StandardScaler()
scaled_features = scaler.fit_transform(features)

scaled_df = pd.DataFrame(scaled_features, columns=features.columns)
scaled_df.head()

Elbow Method , for Choosing the Number of Clusters.

In [ ]:
inertia_values = []
k_range = range(2, 11)

for k in k_range:
    km = KMeans(n_clusters=k, init="k-means++", random_state=42, n_init=10)
    km.fit(scaled_features)
    inertia_values.append(km.inertia_)

plt.figure(figsize=(8, 5))
plt.plot(list(k_range), inertia_values, marker="o")
plt.title("Elbow Method for Optimal k")
plt.xlabel("Number of Clusters (k)")
plt.ylabel("Inertia (Within-Cluster Sum of Squares)")
plt.xticks(list(k_range))
plt.grid(True)
plt.show()

The elbow plot bends clearly around k=3, so going with that as the baseline.

In [ ]:
best_k = 3

kmeans = KMeans(n_clusters=best_k, init="k-means++", random_state=42, n_init=10)
kmeans_labels = kmeans.fit_predict(scaled_features)

df["KMeans_Cluster"] = kmeans_labels
df[["country", "KMeans_Cluster"]].head()

Evaluate K-Means with Silhouatte Score.

In [ ]:
sil_score = silhouette_score(scaled_features, kmeans_labels)
print(f"Silhouette Score for K-Means (k={best_k}): {sil_score:.4f}")

Secondary Model - DBSCAN

In [ ]:
dbscan = DBSCAN(eps=1.5, min_samples=5)
dbscan_labels = dbscan.fit_predict(scaled_features)

df["DBSCAN_Cluster"] = dbscan_labels

n_clusters_db = len(set(dbscan_labels)) - (1 if -1 in dbscan_labels else 0)
n_noise = list(dbscan_labels).count(-1)

print(f"DBSCAN clusters found: {n_clusters_db}")
print(f"Noise points (outliers): {n_noise}")
df[["country", "DBSCAN_Cluster"]].head()

PCA - 2D Visualization of Clusters.

In [ ]:
pca = PCA(n_components=2, random_state=42)
pca_components = pca.fit_transform(scaled_features)

df["PCA1"] = pca_components[:, 0]
df["PCA2"] = pca_components[:, 1]

print(f"Explained variance ratio: {pca.explained_variance_ratio_}")
print(f"Total variance explained by 2 components: {pca.explained_variance_ratio_.sum():.2%}")

plt.figure(figsize=(9, 7))
sns.scatterplot(
    data=df, x="PCA1", y="PCA2",
    hue="KMeans_Cluster", palette="Set1", s=80, edgecolor="black"
)
plt.title("K-Means Clusters Visualized via PCA (2D Projection)")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.legend(title="Cluster")
plt.show()

Cluster Profiles
Average values of each socio-economic/health feature per K-Means cluster , this helps interpret what each cluster represents.

In [ ]:
cluster_profile = df.groupby("KMeans_Cluster")[numeric_cols].mean().round(2)
cluster_profile["num_countries"] = df["KMeans_Cluster"].value_counts()
cluster_profile

In [ ]:
priority_cluster = cluster_profile["child_mort"].idxmax()
print(f"Cluster {priority_cluster} appears to need the most aid (highest avg child mortality).")
df[df["KMeans_Cluster"] == priority_cluster][["country", "child_mort", "income", "gdpp"]] \
    .sort_values("child_mort", ascending=False).head(15)

**Obseravations**

After running K-Means with k=3 and looking at the cluster profile table above, here's
what I'm seeing:
**1. Cluster 1 is clearly the "high-need" group.**
This cluster has by far the highest average child mortality (93 deaths per 1000,
vs 5 and 22 for the other two clusters), the lowest income ($3,942) and the
lowest  GDP per capita ($1,922). Life expectancy here is also the lowest (59 years)
and total fertility is the highest (5 children per woman). This group has 47
countries, and when I sorted by child_mort, the top entries are countries like
Haiti, Sierra Leone, Chad, Central African Republic, Mali, Nigeria, etc. -- which
matches what I'd expect, these are countries that are frequently in the news for
humanitarian crises. **This is the cluster HELP International should prioritize.**

**2. Cluster 0 is the developed/high-income group.**
36 countries fall here, with very low child mortality (5), the highest income
($45,672) and highest GDP per capita ($42,494), plus the best life expectancy
(80 years) and lowest fertility (1.75). These are clearly the wealthy, developed
nations -- aid isn't a priority here.

**3. Cluster 2 is the "in-between" / developing group.**
The largest cluster (84 countries) sits in the middle on almost every metric --
income around $12,300, child mortality around 22, life expectancy around 73. These
are middle-income / emerging economies that are doing okay overall but could still
use some support depending on individual country circumstances.

**4. DBSCAN doesn't really help here.**
With eps=1.5 and min_samples=5, DBSCAN only found 1 real cluster and marked 30
countries as noise (-1). Instead of splitting countries into meaningful groups like
K-Means did, it basically just separated "typical" countries from "extreme" ones
(both very rich and very poor countries probably ended up as noise since they're
far from the dense middle). For this problem, K-Means with k=3 gives a much more
useful/interpretable result than DBSCAN.

**5. PCA plot shows decent separation, but not perfectly clean.**
The 2D PCA plot (63% variance explained) shows the three clusters are mostly
separated along PC1, but there's some overlap at the edges -- which lines up with
the silhouette score of 0.28 (not super high, but reasonable given this is
real-world data, not artificially separated blobs).

---
**Conclusion / Recommendation for HELP International:**
Based on this clustering, the CEO should focus the $10 million budget on the
countries in **Cluster 1** -- this group has the worst combination of high child
mortality, low income, low GDP per capita, and low life expectancy, meaning these
are the countries in the most direst need of aid. A few countries from Cluster 2
that are borderline (closer to Cluster 1's averages) could also be considered as
secondary candidates if budget allows.